<a href="https://colab.research.google.com/github/Hansini23/Statistical-Learning-e23291/blob/main/ME2050_Assignment_7d.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Structural Health Monitoring — Bounded Bayesian Grid Updates

*Original worked solution (own derivation and implementation).*

**Setup recap.** A structural component has a hidden "remaining stiffness efficiency" $\theta\in(0,1]$. A sensor at inspection $k$ reports

$$y_k=\theta\,K_{\text{nominal}}\,e^{\epsilon_k},\qquad \epsilon_k\stackrel{iid}{\sim}\mathscr N(0,\sigma^2),$$

so noise enters *multiplicatively* in the original units but *additively* once we take logs — this is exactly what makes the observation model log-normal. The posterior computed after reading $k-1$ is recycled as the prior going into reading $k$, giving a fully sequential (online) estimator.

## 1. Choosing and Characterizing the Prior

I model the pre-inspection belief with $\Theta\sim\text{Beta}(a_0,b_0)$, $a_0=8,\ b_0=1.5$, restricted in practice to the physically meaningful window $\theta\in[0.01,1]$:

$$f_\Theta^{(0)}(\theta)=\frac{1}{B(a_0,b_0)}\,\theta^{a_0-1}(1-\theta)^{b_0-1}.$$

**Closed-form summary statistics.**

$$\mathbb E[\Theta^{(0)}]=\frac{a_0}{a_0+b_0}=\frac{8}{9.5}=0.8\overline{421},\qquad
\operatorname{Var}[\Theta^{(0)}]=\frac{a_0 b_0}{(a_0+b_0)^2(a_0+b_0+1)}=\frac{12}{90.25\times10.5}\approx0.01265.$$

So the prior standard deviation is roughly $0.112$ — fairly tight, centered high.

**Why a right-skewed Beta and not, say, a Gaussian or a Uniform:**

* Any credible prior for a physical efficiency ratio must vanish for $\theta<0$ and $\theta>1$ automatically — a Gaussian prior would need artificial truncation, whereas Beta support is $[0,1]$ by construction.
* Setting $a_0>b_0$ pushes the bulk of the density toward $\theta=1$, encoding an engineering judgment that "as-manufactured / freshly inspected" components are almost certainly close to full stiffness — a Uniform(0,1) prior would be needlessly agnostic and would slow early convergence unnecessarily when the component really is healthy.
* Keeping $b_0=1.5>1$ (rather than, say, $b_0=1$, i.e. a one-sided power-law prior touching $1$ with infinite density) keeps the density smooth and finite at $\theta=1$, which is numerically friendlier for the grid method used below, while still leaving a genuine (if thin) tail toward lower stiffness so real damage isn't excluded a priori.

In [1]:
import numpy as np
from scipy.stats import beta as beta_dist, lognorm
import plotly.graph_objects as go

A0, B0 = 8.0, 1.5

theta_axis = np.linspace(0.01, 1.0, 400)
prior_pdf_vals = beta_dist.pdf(theta_axis, A0, B0)

mean_closed_form = A0 / (A0 + B0)
var_closed_form = (A0 * B0) / ((A0 + B0) ** 2 * (A0 + B0 + 1))
print(f"E[Theta_0]   = {mean_closed_form:.4f}")
print(f"Var[Theta_0] = {var_closed_form:.5f}  (sd = {var_closed_form**0.5:.4f})")

prior_fig = go.Figure()
prior_fig.add_trace(go.Scatter(
    x=theta_axis, y=prior_pdf_vals, fill='tozeroy',
    line=dict(color='#7B4B94', width=3),
    name='f_Theta^(0)(theta) = Beta(8, 1.5)'
))
prior_fig.add_vline(x=mean_closed_form, line_dash='dash', line_color='black',
                     annotation_text=f"mean={mean_closed_form:.3f}")
prior_fig.update_layout(
    title="Engineering Prior on Remaining Stiffness Efficiency",
    xaxis_title="theta (stiffness efficiency)", yaxis_title="density",
    template="simple_white", xaxis_range=[0.01, 1.0]
)
prior_fig.show()

E[Theta_0]   = 0.8421
Var[Theta_0] = 0.01266  (sd = 0.1125)


## 2. Likelihood of a Sensor Reading

Because $\ln y_k = \ln\theta + \ln K_{\text{nominal}} + \epsilon_k$ and $\epsilon_k\sim\mathscr N(0,\sigma^2)$, the random variable $\ln y_k$ is Gaussian with mean $\ln(\theta K_{\text{nominal}})$ and variance $\sigma^2$. Transforming back (standard log-normal change-of-variables, picking up a Jacobian factor $1/y_k$):

$$\boxed{L(y_k\mid\theta)=\frac{1}{y_k\sigma\sqrt{2\pi}}\exp\left[-\frac{\big(\ln y_k-\ln\theta-\ln K_{\text{nominal}}\big)^2}{2\sigma^2}\right]}, \qquad y_k>0,\ \theta\in(0,1].$$

Given conditional independence of the noise draws across inspections, the likelihood of the *entire* observed trajectory $\mathbf y^{(k)}=(y_1,\dots,y_k)$ is simply the running product:

$$L(\mathbf y^{(k)}\mid\theta)=\prod_{i=1}^{k}L(y_i\mid\theta)
=\left[\prod_{i=1}^k \frac{1}{y_i}\right]\,(2\pi\sigma^2)^{-k/2}\,
\exp\left[-\frac{1}{2\sigma^2}\sum_{i=1}^{k}\big(\ln y_i-\ln\theta-\ln K_{\text{nominal}}\big)^2\right].$$

In practice, because this product of $k$ densities can underflow numerically for larger $k$, it is safer to work with the **sum of log-likelihoods**,
$$\ell(\mathbf y^{(k)}\mid\theta)=\sum_{i=1}^{k}\ln L(y_i\mid\theta),$$
and exponentiate only once at the end — this is the strategy used in the sequential update loop below.

## 3. Why Conjugacy Fails, and the Recursive Update Rule

A Beta prior is conjugate only to likelihoods that are themselves powers of $\theta$ and $(1-\theta)$ (as in the Binomial case). Here the likelihood's dependence on $\theta$ sits inside a *squared logarithm* wrapped in an exponential — $\exp[-(\ln y_k-\ln\theta-\ln K)^2/2\sigma^2]$ — which cannot be rearranged into the form $\theta^{p}(1-\theta)^{q}$ for constants $p,q$. Multiplying it against a Beta density therefore produces a new function of $\theta$ that is **not itself a Beta density** (nor any other standard named family): no finite set of sufficient statistics summarizes the posterior shape, so it must be tracked as a full curve.

The recursion is nevertheless simple to state, since Bayes' rule always holds pointwise in $\theta$:

$$f_{\Theta\mid\mathbf Y^{(k)}}(\theta\mid\mathbf y^{(k)})=\frac{L(y_k\mid\theta)\,f_{\Theta\mid\mathbf Y^{(k-1)}}(\theta\mid\mathbf y^{(k-1)})}{\displaystyle\int_0^1 L(y_k\mid s)\,f_{\Theta\mid\mathbf Y^{(k-1)}}(s\mid\mathbf y^{(k-1)})\,ds},\qquad k=1,2,\dots,n,$$

initialized at $f_{\Theta\mid\mathbf Y^{(0)}}\equiv f_\Theta^{(0)}=\text{Beta}(8,1.5)$. Equivalently, up to the (theta-independent) normalizing constant $Z_k$ in the denominator,

$$f_{\Theta\mid\mathbf Y^{(k)}}(\theta\mid\mathbf y^{(k)}) \;\propto\; L(y_k\mid\theta)\cdot f_{\Theta\mid\mathbf Y^{(k-1)}}(\theta\mid\mathbf y^{(k-1)}).$$

## 4. Point Estimators as Integrals

With no analytic posterior available, both classical point estimates reduce to integrals over the bounded parameter space $(0,1]$ that must be approximated numerically:

$$\widehat\theta_{\text{Bayes}}^{(k)}=\mathbb E\!\left[\Theta\mid\mathbf Y^{(k)}=\mathbf y^{(k)}\right]=\int_0^1\theta\, f_{\Theta\mid\mathbf Y^{(k)}}(\theta\mid\mathbf y^{(k)})\,d\theta \qquad\text{(minimizer of expected squared error)},$$

$$\widehat\theta_{\text{MAP}}^{(k)}=\underset{\theta\in(0,1]}{\arg\max}\; f_{\Theta\mid\mathbf Y^{(k)}}(\theta\mid\mathbf y^{(k)}) \qquad\text{(the single most probable stiffness level given all evidence so far)}.$$

## 5. Grid-Based Numerical Scheme

Rather than tracking a formula, I track $M$ numbers — the posterior density evaluated at fixed grid nodes — and refresh them every time a new reading arrives.

1. **Discretize the domain.** Lay down $M$ evenly spaced nodes $\theta_1,\dots,\theta_M$ covering $[\theta_{lo},\theta_{hi}]=[0.01,1.0]$ with spacing $\Delta\theta=(\theta_{hi}-\theta_{lo})/(M-1)$. The small offset from $0$ sidesteps the $\ln(0)$ singularity in the log-normal likelihood and reflects that $\theta$ exactly at $0$ has no physical meaning for a still-standing structure; $\theta=1$ is kept as an attainable endpoint.
2. **Seed the grid with the prior.** Evaluate $f_\Theta^{(0)}(\theta_m)$ at every node, then rescale the whole array so its area (via the trapezoid rule) is exactly $1$.
3. **Fold in each new reading.** On arrival of $y_k$:
   * evaluate the log-likelihood $\ell_m=\ln L(y_k\mid\theta_m)$ at every node (numerically stable),
   * multiply: work in log-space, $\log\tilde P_k(\theta_m)=\log P_{k-1}(\theta_m)+\ell_m$, then exponentiate after subtracting the running max for stability,
   * **renormalize with the composite trapezoid rule**:
     $$Z_k=\sum_{m=1}^{M-1}\tfrac12\big[\tilde P_k(\theta_m)+\tilde P_k(\theta_{m+1})\big]\Delta\theta \;=\;\texttt{np.trapezoid}(\tilde P_k,\theta),\qquad P_k(\theta_m)=\tilde P_k(\theta_m)/Z_k.$$
4. **Read off estimators.** $\widehat\theta_{\text{Bayes}}^{(k)}=\texttt{np.trapezoid}(\theta\cdot P_k,\theta)$; $\widehat\theta_{\text{MAP}}^{(k)}=\theta_{m^\*}$ for $m^\*=\arg\max_m P_k(\theta_m)$.
5. **Recurse.** $P_k$ becomes the seed for step $k+1$; repeat until all $n$ readings are consumed.

## 6. Simulated Monitoring Run ($\theta_{\text{true}}=0.68$, impact scenario)

Below I implement the scheme as a small self-contained class (`SHMTracker`) rather than a flat script, and drive it through $n=15$ inspections after a sudden impact drops true stiffness efficiency to $0.68$ (`K_nominal = 50.0` kN/mm, `sigma = 0.15`).

In [2]:
import numpy as np
from scipy.stats import lognorm
import plotly.graph_objects as go

class SHMTracker:
    '''Sequential bounded-grid Bayesian tracker for stiffness efficiency theta.'''

    def __init__(self, K_nominal, sigma, a0=8.0, b0=1.5, grid_lo=0.01, grid_hi=1.0, n_grid=400):
        self.K_nominal = K_nominal
        self.sigma = sigma
        self.theta = np.linspace(grid_lo, grid_hi, n_grid)
        prior_vals = beta_dist.pdf(self.theta, a0, b0)
        self.posterior = prior_vals / np.trapezoid(prior_vals, self.theta)
        self.history_mean = [self._posterior_mean()]
        self.history_map = [self._posterior_map()]
        self.snapshots = {0: self.posterior.copy()}

    def _posterior_mean(self):
        return np.trapezoid(self.theta * self.posterior, self.theta)

    def _posterior_map(self):
        return self.theta[np.argmax(self.posterior)]

    def update(self, y_k, step_index):
        scale = self.theta * self.K_nominal
        log_lik = lognorm.logpdf(y_k, s=self.sigma, scale=scale)
        log_unnorm = np.log(self.posterior + 1e-300) + log_lik
        log_unnorm -= log_unnorm.max()          # stability shift
        unnorm = np.exp(log_unnorm)
        Z = np.trapezoid(unnorm, self.theta)
        self.posterior = unnorm / Z
        self.history_mean.append(self._posterior_mean())
        self.history_map.append(self._posterior_map())
        self.snapshots[step_index] = self.posterior.copy()


# ---- Simulation configuration ----
rng = np.random.default_rng(seed=7)
THETA_TRUE = 0.68
K_NOM = 50.0
SIGMA = 0.15
N_STEPS = 15
snap_steps = {0, 1, 2, 5, 10, 15}

tracker = SHMTracker(K_nominal=K_NOM, sigma=SIGMA)
observed_readings = []

for step in range(1, N_STEPS + 1):
    eps = rng.normal(0.0, SIGMA)
    y_obs = THETA_TRUE * K_NOM * np.exp(eps)
    observed_readings.append(y_obs)
    tracker.update(y_obs, step)

print("Simulated sensor stream (kN/mm):")
print(np.round(observed_readings, 2))
print(f"\nFinal running Bayes estimate  (k=15): {tracker.history_mean[-1]:.4f}")
print(f"Final running MAP estimate   (k=15): {tracker.history_map[-1]:.4f}")

Simulated sensor stream (kN/mm):
[34.01 35.56 32.63 29.75 31.76 29.3  34.31 41.57 31.58 30.98 36.59 35.87
 34.54 29.57 33.85]

Final running Bayes estimate  (k=15): 0.6738
Final running MAP estimate   (k=15): 0.6725


In [3]:
# ---- Plot 1: posterior shape evolution at chosen milestones ----
density_fig = go.Figure()
palette = ['#B0B0B0', '#E69F00', '#56B4E9', '#009E73', '#D55E00', '#CC79A7']
for color, step in zip(palette, sorted(snap_steps)):
    label = "Prior (step 0)" if step == 0 else f"After reading #{step}"
    style = dict(dash='dash', width=2.5) if step == 0 else dict(width=2.5)
    density_fig.add_trace(go.Scatter(
        x=tracker.theta, y=tracker.snapshots[step], mode='lines',
        name=label, line=dict(color=color, **style)
    ))

density_fig.add_vline(x=THETA_TRUE, line_color='crimson', line_dash='dot', line_width=2,
                       annotation_text=f"true theta = {THETA_TRUE}", annotation_position="top left")
density_fig.update_layout(
    title="Posterior Shape Evolution After a Simulated Impact Event",
    xaxis_title="theta (remaining stiffness efficiency)",
    yaxis_title="posterior density",
    template="simple_white",
    legend=dict(x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.6)")
)
density_fig.show()

# ---- Plot 2: point-estimate convergence timeline ----
steps_axis = list(range(N_STEPS + 1))
timeline_fig = go.Figure()
timeline_fig.add_hline(y=THETA_TRUE, line_color='crimson', line_dash='dash',
                        annotation_text=f"true theta = {THETA_TRUE}")
timeline_fig.add_trace(go.Scatter(x=steps_axis, y=tracker.history_mean, mode='lines+markers',
                                   name='running posterior mean', line=dict(color='#2166AC', width=2.5)))
timeline_fig.add_trace(go.Scatter(x=steps_axis, y=tracker.history_map, mode='lines+markers',
                                   name='running MAP', line=dict(color='#1B7837', width=2, dash='dot')))
timeline_fig.update_layout(
    title="Convergence of Sequential Estimators Toward the True Post-Impact State",
    xaxis_title="inspection step k",
    yaxis_title="theta_hat",
    template="simple_white"
)
timeline_fig.show()

# ---- Convergence diagnostics ----
band = 0.03
mean_arr = np.array(tracker.history_mean)
first_lock_in = next((k for k in range(len(mean_arr))
                       if np.all(np.abs(mean_arr[k:] - THETA_TRUE) < band)), None)
print(f"Running mean settles permanently within +/-{band} of theta_true starting at step {first_lock_in}.")

Running mean settles permanently within +/-0.03 of theta_true starting at step 4.


### Reading the results

* **The optimistic prior gets overridden quickly.** The Beta(8,1.5) prior starts centered near $0.84$, well above the actual post-impact value of $0.68$. Because each log-normal likelihood term is fairly peaked (moderate $\sigma=0.15$), just a handful of readings is enough to drag the posterior mass down toward the true value — in this run the mean and MAP are already hovering close to $0.68$ within the first 2–3 readings, and the diagnostic above reports the step at which the estimate permanently stays inside a $\pm0.03$ band of the true value.
* **Density sharpening = growing confidence.** Comparing the step-0 curve to the step-15 curve in the first figure, the posterior visibly narrows and gains height as more independent evidence accumulates — a direct visual signature of shrinking posterior variance.
* **Engineering takeaway.** For safety-critical thresholds (e.g., "flag for maintenance if $\theta$ credibly falls below 0.75"), a wide early posterior would force conservative, uncertain decision-making, while the narrow late-stage posterior lets an automated SHM system commit to an alert with high confidence after relatively few inspections — demonstrating the practical payoff of updating beliefs sequentially rather than waiting to pool all data and estimate once.